In [0]:
staging_table = "dbacademy.Superstore.fact_sales_staging_pos"

current_count = spark.table(staging_table).count()
print(f"Current row count: {current_count}")

MIN_EXPECTED_ROWS = 500

if current_count < MIN_EXPECTED_ROWS:
    raise Exception(f"DATA QUALITY FAILURE: Row count dropped to {current_count}, below minimum threshold of {MIN_EXPECTED_ROWS}")

print("✓ Check 1 passed: row count is healthy")

In [0]:
from pyspark.sql.functions import col

null_check = spark.table(staging_table).filter(
    col("product_id").isNull() |
    col("customer_id").isNull() |
    col("postal_code").isNull()
).count()

print(f"Γραμμές με null σε κρίσιμα πεδία: {null_check}")

if null_check > 0:
    raise Exception(f"DATA QUALITY FAILURE: Found {null_check} rows with null product_id/customer_id/postal_code")

print("✓ Check 2 passed: no nulls in critical fields")

In [0]:
total_rows = spark.table(staging_table).count()
distinct_receipt_ids = spark.table(staging_table).select("receipt_id").distinct().count()

print(f"Total rows: {total_rows}, Distinct receipt_ids: {distinct_receipt_ids}")

if total_rows != distinct_receipt_ids:
    duplicate_count = total_rows - distinct_receipt_ids
    raise Exception(f"DATA QUALITY FAILURE: Found {duplicate_count} duplicate receipt_id(s)")

print("✓ Check 3 passed: no duplicate receipt_ids")